In [0]:
%sql
-- create the HTTP connection to fetch the data 2 way to create first is using UI and second is using query


create connection if not exists earthquake_test type HTTP
OPTIONS(
    host = 'https://earthquake.usgs.gov',
    base_path ='/earthquakes/feed/v1.0/',
    port = '443',
    bearer_token = 'na'
)

In [0]:
from databricks.sdk import WorkspaceClient


w = WorkspaceClient()

conn = w.connections.get('earthquake_data')
base_url = f"{conn.options['host']}{conn.options['base_path']}"
print(base_url)

In [0]:
dbutils.widgets.text('catalog','dlt_pipeline_dev','catalog')
catalog_name = dbutils.widgets.get('catalog')


print(catalog_name)

In [0]:
# Create volume to store the data - volume can store structured and semi-structured data

spark.sql(f"USE catalog `{catalog_name}` ")
spark.sql(f'use schema bronze')
spark.sql("CREATE VOLUME IF NOT EXISTS earthquake_data")

In [0]:
import requests
import json
import datetime

# url = 'https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson'

## instead of using above thing we will use workspaceClient connection

url = f"{base_url}/summary/all_day.geojson"
response = requests.get(url)
if response.status_code != 200:
    raise Exception(f"Error while fetching data from api {url} ")
data = response.json()

current_date = datetime.datetime.now().strftime("%Y-%m-%d")
# dbutils.fs.put('/Volumes/etl_pipeline_dlt/bronze/earthquake_data',json.dumps(data),overwrite = True)

dbutils.fs.put(
    f"/Volumes/{catalog_name}/bronze/earthquake_data/earthquake_data_{current_date}.json",
    json.dumps(data),
    overwrite=True,
)